# 01. Echoes 데이터 품질 확인

Echoes manifest의 유형·생성기·장르 분포와 실제 파일을 대조한다. TTA 중복 경로 3행은 원곡 대응을 알 수 없어 제외했다. 이 단계의 목적은 이후 사용할 FAKE 집합을 확정하는 것이다.

## 0. 프로젝트 경로와 데이터 구조

```text
project/
├── data/
│   ├── raw/
│   │   ├── Echoes/Echoes/
│   │   │   ├── dataset_manifest.csv
│   │   │   ├── TTA/
│   │   │   └── ATA/
│   │   └── FMA/fma_metadata/
│   │       ├── tracks.csv
│   │       ├── genres.csv
│   │       └── ...
│   ├── metadata/
│   └── processed/
├── notebooks/
├── src/
├── experiments/
├── results/
└── checkpoints/
```

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

ECHOES_ROOT = PROJECT_ROOT / "data/raw/Echoes/Echoes"
ECHOES_MANIFEST = ECHOES_ROOT / "dataset_manifest.csv"

FMA_META_ROOT = PROJECT_ROOT / "data/raw/FMA/fma_metadata"
FMA_TRACKS = FMA_META_ROOT / "tracks.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Echoes manifest exists:", ECHOES_MANIFEST.exists())
print("FMA tracks.csv exists:", FMA_TRACKS.exists())

PROJECT_ROOT: /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project
Echoes manifest exists: True
FMA tracks.csv exists: True


## 1. Echoes manifest 기본 구조 확인

확인된 컬럼:
- `path_in_dataset`
- `original_audio`
- `generator`
- `type`
- `genre`
- `description`
- `duration`

In [2]:
echoes = pd.read_csv(ECHOES_MANIFEST)

print("shape:", echoes.shape)
print("\ncolumns:")
print(echoes.columns.tolist())

display(echoes.head(3))

shape: (4468, 7)

columns:
['path_in_dataset', 'original_audio', 'generator', 'type', 'genre', 'description', 'duration']


,path_in_dataset,original_audio,generator,type,genre,description,duration
0,TTA/acestep/10000_People_Chanting_Im_an_Indivi...,"10,000 People Chanting, ""I'm an Individual"" - ...",acestep,TTA,Electronic,"cinematic, idm, downtempo, layered, swelling, ...",45.70
1,TTA/acestep/1984_Punk_Rock_Opera_acestep_TTA_0...,1984 - Punk Rock Opera,acestep,TTA,Rock,"hardcore-punk, political, d-beat, shouted-chor...",54.80
2,TTA/acestep/2Much_Andy_Spinelli_Alex_Sánchez_H...,2Much (Andy Spinelli & Alex Sánchez House Edit...,acestep,TTA,Electronic,"house, four-on-the-floor, piano-stabs, filtere...",80.71


In [3]:
# Echoes manifest 기본 구조 확인
print("===== TYPE =====")
print(echoes["type"].value_counts(dropna=False))

print("\n===== GENERATOR =====")
print(echoes["generator"].value_counts(dropna=False))

print("\n===== GENRE =====")
print(echoes["genre"].value_counts(dropna=False))

===== TYPE =====
type
TTA    3165
ATA    1303
Name: count, dtype: int64

===== GENERATOR =====
generator
diffrhythm     594
musicgen       591
audioldm       587
songgen        561
producer       300
suno           300
udio           300
elevenlabs     300
brev           298
acestep        294
stableaudio    194
mubert         149
Name: count, dtype: int64

===== GENRE =====
genre
Electronic    1653
Rock          1594
Pop           1221
Name: count, dtype: int64


### 실제 확인 결과

**TYPE**
- TTA: 3,165
- ATA: 1,303

**GENRE**
- Electronic: 1,653
- Rock: 1,594
- Pop: 1,221

생성기 전체는 12종이다.

## 2. TTA만 필터링

In [4]:
# TTA만 필터링
tta = echoes[echoes["type"] == "TTA"].copy()

print("TTA rows:", len(tta))
print("Unique original_audio:", tta["original_audio"].nunique())
print("Unique generators:", tta["generator"].nunique())

print("\n===== TTA GENERATOR =====")
print(tta["generator"].value_counts())

print("\n===== TTA GENRE =====")
print(tta["genre"].value_counts())

print("\n===== DURATION =====")
print(tta["duration"].describe())

TTA rows: 3165
Unique original_audio: 296
Unique generators: 12

===== TTA GENERATOR =====
generator
suno           300
udio           300
elevenlabs     300
diffrhythm     299
brev           298
musicgen       296
acestep        294
audioldm       292
songgen        292
stableaudio    194
producer       151
mubert         149
Name: count, dtype: int64

===== TTA GENRE =====
genre
Electronic    1131
Rock          1130
Pop            904
Name: count, dtype: int64

===== DURATION =====
count    3165.000000
mean      121.362408
std        74.902808
min        17.960000
25%        30.720000
50%       130.860000
75%       180.000000
max       479.960000
Name: duration, dtype: float64


## 3. TTA 중복 파일 경로 검사

MusicGen TTA에서 하나의 실제 파일이 서로 다른 원곡 3개에 연결된 충돌이 확인되었다.

In [5]:
# TTA 중복 파일 경로 검사
tta_dup = tta[tta["path_in_dataset"].duplicated(keep=False)].sort_values(
    "path_in_dataset"
)

print("Duplicated TTA rows:", len(tta_dup))
print("Duplicated TTA unique paths:", tta_dup["path_in_dataset"].nunique())

display(tta_dup[["path_in_dataset", "original_audio", "generator", "genre"]])

Duplicated TTA rows: 3
Duplicated TTA unique paths: 1


,path_in_dataset,original_audio,generator,genre
4454,TTA/musicgen/_musicgen_TTA_001.wav,В Наших Сердцах - Чокнутый Пропеллер,musicgen,Rock
4456,TTA/musicgen/_musicgen_TTA_001.wav,Глазами Детей - Чокнутый Пропеллер,musicgen,Rock
4458,TTA/musicgen/_musicgen_TTA_001.wav,Кортни Лав - Чокнутый Пропеллер,musicgen,Rock


### 확인된 충돌

동일 파일:

```text
TTA/musicgen/_musicgen_TTA_001.wav
```

서로 다른 원곡 3개가 같은 파일을 참조하므로 어느 원곡과 실제로 대응되는지 확정할 수 없다.  
따라서 **3행 모두 제외**한다.

## 4. Clean TTA 생성 및 실제 파일 존재 확인

In [6]:
# Clean TTA 생성 및 실제 파일 존재 확인
dup_mask = tta["path_in_dataset"].duplicated(keep=False)
tta_clean = tta[~dup_mask].copy()

tta_clean["full_path"] = tta_clean["path_in_dataset"].apply(lambda x: ECHOES_ROOT / x)
tta_clean["exists"] = tta_clean["full_path"].apply(lambda p: p.exists())

print("===== CLEAN TTA =====")
print("Original TTA rows :", len(tta))
print("Excluded rows     :", int(dup_mask.sum()))
print("Clean TTA rows    :", len(tta_clean))
print("Generators        :", tta_clean["generator"].nunique())
print("Original groups   :", tta_clean["original_audio"].nunique())

print("\n===== GENERATOR COUNTS =====")
print(tta_clean["generator"].value_counts())

print("\n===== GENRE COUNTS =====")
print(tta_clean["genre"].value_counts())

print("\n===== FILE CHECK =====")
print("Existing files :", int(tta_clean["exists"].sum()))
print("Missing files  :", int((~tta_clean["exists"]).sum()))

===== CLEAN TTA =====
Original TTA rows : 3165
Excluded rows     : 3
Clean TTA rows    : 3162
Generators        : 12
Original groups   : 296

===== GENERATOR COUNTS =====
generator
suno           300
udio           300
elevenlabs     300
diffrhythm     299
brev           298
acestep        294
musicgen       293
audioldm       292
songgen        292
stableaudio    194
producer       151
mubert         149
Name: count, dtype: int64

===== GENRE COUNTS =====
genre
Electronic    1131
Rock          1127
Pop            904
Name: count, dtype: int64

===== FILE CHECK =====
Existing files : 3162
Missing files  : 0


### Clean TTA 최종 결과

- 원본 TTA: **3,165**
- 중복 충돌 제외: **3**
- Clean TTA: **3,162**
- 생성기: **12종**
- `original_audio` 그룹: **296개**
- 실제 파일 존재: **3,162 / 3,162**
- Missing file: **0**

생성기별 Clean TTA:
- suno 300
- udio 300
- elevenlabs 300
- diffrhythm 299
- brev 298
- acestep 294
- musicgen 293
- audioldm 292
- songgen 292
- stableaudio 194
- producer 151
- mubert 149

장르별:
- Electronic 1,131
- Rock 1,127
- Pop 904

## 5. Manifest와 실제 오디오 파일 비교

In [7]:
# Manifest와 실제 오디오 파일 비교
manifest_paths = set(
    echoes["path_in_dataset"].astype(str).str.replace("\\", "/", regex=False)
)

audio_exts = {".wav", ".mp3", ".flac"}
actual_paths = set()

for p in ECHOES_ROOT.rglob("*"):
    if p.is_file() and p.suffix.lower() in audio_exts:
        actual_paths.add(p.relative_to(ECHOES_ROOT).as_posix())

extra_files = sorted(actual_paths - manifest_paths)
missing_files = sorted(manifest_paths - actual_paths)

print("===== MANIFEST vs ACTUAL AUDIO =====")
print("Manifest rows         :", len(echoes))
print("Unique manifest paths :", len(manifest_paths))
print("Actual audio files    :", len(actual_paths))
print("Extra audio files     :", len(extra_files))
print("Missing audio files   :", len(missing_files))

print("\n===== EXTRA FILES =====")
for x in extra_files:
    print(x)

===== MANIFEST vs ACTUAL AUDIO =====
Manifest rows         : 4468
Unique manifest paths : 4464
Actual audio files    : 4488
Extra audio files     : 24
Missing audio files   : 0

===== EXTRA FILES =====
ATA/songgen/1984_Punk_Rock_Opera_songgen_ATA_001.mp3
ATA/songgen/2_Wasnt_There_Isle_of_Pine_songgen_ATA_001.mp3
ATA/songgen/50000_Volts_of_Democracy_mp3_Legally_Blind_songgen_ATA_001.mp3
ATA/songgen/Attention_Pete_Prodoehl_songgen_ATA_001.mp3
ATA/songgen/Bergwald_Bergwald_Every_Now_and_Every_Then_songgen_ATA_001.mp3
ATA/songgen/Fear_Los_Fancy_Free_songgen_ATA_001.mp3
ATA/songgen/Five_40_DerbySlow_Whistle_The_Crypts_songgen_ATA_001.mp3
ATA/songgen/Gloomy_Sunday_AmortE_songgen_ATA_001.mp3
ATA/songgen/I_Dream_So_Vividly_Uninhabitable_Mansions_songgen_ATA_001.mp3
ATA/songgen/KISS_my_Boots_The_Zombie_Dandies_songgen_ATA_001.mp3
ATA/songgen/Leaving_Here_Mod_Fun_songgen_ATA_001.mp3
ATA/songgen/March_of_the_GPA_Mechanics_tghost_songgen_ATA_001.mp3
ATA/songgen/Mettle_Pipe_Choir_songgen_ATA_001.mp

### 확인 결과

- Manifest rows: **4,468**
- Unique manifest paths: **4,464**
- Actual audio files: **4,488**
- Extra audio files: **24**
- Missing audio files: **0**

Extra 24개:
- ATA/songgen: 23개
- TTA/acestep: 1개

Manifest에 없는 extra 파일은 metadata 연결을 보장할 수 없으므로 연구 데이터에 포함하지 않는다.

## 6. 전체 manifest 중복 경로 검사

In [8]:
# 전체 manifest 중복 경로 검사
all_dup = echoes[echoes["path_in_dataset"].duplicated(keep=False)].sort_values(
    "path_in_dataset"
)

print("Duplicated rows        :", len(all_dup))
print("Duplicated unique paths:", all_dup["path_in_dataset"].nunique())

display(all_dup[["path_in_dataset", "original_audio", "generator", "type", "genre"]])

Duplicated rows        : 6
Duplicated unique paths: 2


,path_in_dataset,original_audio,generator,type,genre
4455,ATA/musicgen/_musicgen_ATA_001.wav,В Наших Сердцах - Чокнутый Пропеллер,musicgen,ATA,Rock
4457,ATA/musicgen/_musicgen_ATA_001.wav,Глазами Детей - Чокнутый Пропеллер,musicgen,ATA,Rock
4459,ATA/musicgen/_musicgen_ATA_001.wav,Кортни Лав - Чокнутый Пропеллер,musicgen,ATA,Rock
4454,TTA/musicgen/_musicgen_TTA_001.wav,В Наших Сердцах - Чокнутый Пропеллер,musicgen,TTA,Rock
4456,TTA/musicgen/_musicgen_TTA_001.wav,Глазами Детей - Чокнутый Пропеллер,musicgen,TTA,Rock
4458,TTA/musicgen/_musicgen_TTA_001.wav,Кортни Лав - Чокнутый Пропеллер,musicgen,TTA,Rock


### 전체 중복 구조

중복 경로는 2개다.

- `ATA/musicgen/_musicgen_ATA_001.wav`
- `TTA/musicgen/_musicgen_TTA_001.wav`

각 경로가 서로 다른 3개 원곡 행에서 반복되어 전체 manifest에서 중복 초과분은 4행이다.

## 7. FMA metadata 구조 확인

In [9]:
fma = pd.read_csv(FMA_TRACKS, header=[0, 1], index_col=0)

print("FMA shape:", fma.shape)
print("\nFirst 50 columns:")
print(fma.columns.tolist()[:50])

display(fma.head(3))

FMA shape: (106574, 52)

First 50 columns:
[('album', 'comments'), ('album', 'date_created'), ('album', 'date_released'), ('album', 'engineer'), ('album', 'favorites'), ('album', 'id'), ('album', 'information'), ('album', 'listens'), ('album', 'producer'), ('album', 'tags'), ('album', 'title'), ('album', 'tracks'), ('album', 'type'), ('artist', 'active_year_begin'), ('artist', 'active_year_end'), ('artist', 'associated_labels'), ('artist', 'bio'), ('artist', 'comments'), ('artist', 'date_created'), ('artist', 'favorites'), ('artist', 'id'), ('artist', 'latitude'), ('artist', 'location'), ('artist', 'longitude'), ('artist', 'members'), ('artist', 'name'), ('artist', 'related_projects'), ('artist', 'tags'), ('artist', 'website'), ('artist', 'wikipedia_page'), ('set', 'split'), ('set', 'subset'), ('track', 'bit_rate'), ('track', 'comments'), ('track', 'composer'), ('track', 'date_created'), ('track', 'date_recorded'), ('track', 'duration'), ('track', 'favorites'), ('track', 'genre_top'), 

album                                                     \
         comments         date_created        date_released engineer   
track_id                                                               
2               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
3               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
5               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   

                                                         ...       track  \
         favorites id information listens producer tags  ... information   
track_id                                                 ...               
2                4  1     <p></p>    6073      NaN   []  ...         NaN   
3                4  1     <p></p>    6073      NaN   []  ...         NaN   
5                4  1     <p></p>    6073      NaN   []  ...         NaN   

                                 \
         interest language_code   
track_id                          
2            4656            en   
3            1470            en   
5            1933            en   

                                                                              \
                                                    license listens lyricist   
track_id                                                                       
2         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1293      NaN   
3         Attribution-NonCommercial-ShareAlike 3.0 Inter...     514      NaN   
5         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1151      NaN   

                                              
         number publisher tags         title  
track_id                                      
2             3       NaN   []          Food  
3             4       NaN   []  Electric Ave  
5             6       NaN   []    This World  

[3 rows x 52 columns]

### 이후 사용할 FMA 주요 컬럼

- `('artist', 'name')`
- `('track', 'title')`
- `('track', 'genre_top')`
- `('track', 'license')`
- `('track', 'duration')`
- `('set', 'subset')`

FMA 전체 `tracks.csv`에는 **106,574곡**이 있다.

## 8. 현재 결론

### Echoes FAKE 데이터 정제 결과

```text
Echoes 전체 manifest 4,468
        ↓
TTA 3,165 / ATA 1,303
        ↓
ATA 제외
        ↓
MusicGen TTA 동일 파일 충돌 3행 제외
        ↓
Clean TTA = 3,162
        ↓
12 generators
296 original_audio groups
Missing audio = 0
```

현재 연구의 FAKE 데이터는 **Clean TTA 3,162개**를 기준으로 진행한다.

## 정제한 FAKE 목록

- Echoes manifest 4,468행 중 TTA 3,165행, ATA 1,303행을 확인했다.
- MusicGen 경로 충돌 3행을 제외한 Clean TTA는 **3,162개**, `original_audio` 그룹은 **296개**다.
- Clean TTA 실제 파일 누락은 **0개**이며, 12개 generator와 3개 장르를 포함한다.
- Manifest 밖 오디오 24개는 metadata 연결을 보장할 수 없어 연구 데이터에서 제외한다.